# Versionnage et traçage des données

Voyons comment utiliser et tracer des données au cours d'un projet de data science.

De nombreux outils permettent de le faire. Nous utiliserons [DVC](https://dvc.org/) pour cette démonstration.

In [ ]:
# @title ## Configuration & installations
!git config --global user.email "jeanne@durant.fr"
!git config --global user.name "Jeanne Durant"
!git config --global init.defaultBranch main
!apt install tree
!pip install dvc

## Initialisation du dépôt du projet

In [ ]:
!git init
!dvc init

## Récupération des données

Il y a plusieurs manières d'importer des données dans le dépôt du projet. Nous utiliserons ici [`dvc get`](https://dvc.org/doc/command-reference/get), qui récupère des données depuis un dépôt git/DVC, puis nous les ferons suivre par DVC avec [`dvc add`](https://dvc.org/doc/command-reference/add).

In [ ]:
!dvc get https://github.com/shuuchuu/datasets wikipedia-movie-plots/wiki_movie_plots_deduped.csv -o data.csv

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc
!tree .dvc

## Enregistrement des métadonnées dans le dépôt du projet

In [ ]:
!git add .gitignore data.csv.dvc

In [ ]:
!git commit -m "Ajout d'une première version des données"
!git tag "v1"

## Stockage des données sur un serveur

Avec DVC, les données sont stockées à part des métadonnées. Nous configurons ici un faux serveur de stockage sur cette machine, mais ce serait d'habitude un serveur dédié (sur AWS par exemple).

In [ ]:
!dvc remote add -d local /tmp/dvcstore

In [ ]:
!dvc push

## Modification des données et suivi des changements

In [ ]:
!sort -r < data.csv > a && dvc remove data.csv.dvc && mv a data.csv

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc
!tree .dvc

In [ ]:
!git add data.csv.dvc .gitignore

In [ ]:
!git commit -m "Données v2"
!git tag "v2"

In [ ]:
!dvc push

## Exemple d'utilisation des versions : retour en arrière

Imaginons que la nouvelle version des données pose problème. On peut revenir facilement à la première version&nbsp;: c'est tout l'intérêt de versionner ses données.

In [ ]:
!git checkout v1 data.csv.dvc

In [ ]:
!cat data.csv.dvc
!tree .dvc

In [ ]:
!dvc checkout data.csv

On enregistre maintenant dans le dépôt les métadonnées indiquant ce retour en arrière.

In [ ]:
!git add data.csv.dvc

In [ ]:
!git commit -m "Retour aux données v1"
!git tag v3

In [ ]:
!dvc push

## Traitement des données

In [ ]:
%%writefile upper.py
from pathlib import Path
from sys import argv

Path(argv[2]).write_text(
    Path(argv[1]).read_text(encoding="utf8").upper(),
    encoding="utf8")

[`dvc stage add`](https://dvc.org/doc/command-reference/stage/add) ajoute une étape de traitement à nos métadonnées. On saura ainsi exactement quels traitements ont été appliqués et quels résultats ils ont produits.

In [ ]:
!dvc stage add -n transform-uppercase -d data.csv -o data-upper.csv python upper.py data.csv data-upper.csv

[`dvc repro`](https://dvc.org/doc/command-reference/repro) exécute cette étape.

In [ ]:
!dvc repro

In [ ]:
!git add dvc.yaml .gitignore dvc.lock

In [ ]:
!git commit -m "Pipeline pour passer un fichier en majuscule"

In [ ]:
!dvc push